# Эквивалентная сфера: импедансный критерий вместо геометрического

Документ отвечает на два вопроса, возникших при переходе от модели сердца как эквивалентной сферы к работе с реальной геометрией по томографии.

1. Какое значение базового импеданса следует подставлять в модель, если каналы измерительного прибора неравноценны по исправности, и насколько выбор влияет на восстановленный радиус.
2. Будет ли сфера, построенная как геометрически эквивалентная реальному сердцу, эквивалентна ему и по моделируемому импедансу.

Область охвата — модель сердца ([01](01_Математическая_модель_импеданса.md), реализация [02](02_Kandidadskaya.ipynb)), монтаж с фиксированной базой $a=90$ мм. Лёгочный пайплайн (03–12) здесь не затрагивается.

## §0. Методика и словарь терминов

### Решаемая задача
Требуется установить, какие величины должны совпадать у реального сердца и заменяющей его сферы, чтобы замена была допустима в задаче восстановления изменения объёма по импедансу, и какие погрешности вносит замена по геометрическому признаку.

### Метод и обоснование выбора
Возмущение импеданса, вносимое включением, раскладывается в ряд по сферическим гармоникам ([01](01_Математическая_модель_импеданса.md), §3.1). Вклад каждого члена вычисляется отдельно, что позволяет установить, какими свойствами включения определяется наблюдаемая величина. Для несферического включения дипольный отклик описывается тензором поляризуемости, вычисляемым через коэффициенты деполяризации эллипсоида; сопоставление с равнообъёмной сферой даёт оценку погрешности замены.

### Словарь терминов
- **Эквивалентная сфера** — сфера, заменяющая реальное сердце в модели. Требует указания критерия эквивалентности; в настоящем документе различаются геометрическая и импедансная.
- **Геометрическая эквивалентность** — совпадение объёма либо наилучшее приближение контура окружностью.
- **Импедансная эквивалентность** — совпадение вычисляемого моделью импеданса либо его производной по объёму.
- **Мультипольный член** — слагаемое ряда (18.1) с номером $n$: $n=1$ дипольный, $n=2$ квадрупольный.
- **Поляризуемость** — коэффициент пропорциональности между наведённым дипольным моментом включения и внешним полем; для эллипсоида зависит от формы через коэффициенты деполяризации, а не только от объёма.
- **Коэффициент деполяризации** $L_i$ — безразмерная характеристика формы эллипсоида вдоль оси $i$; для шара $L_i=1/3$, сумма по трём осям равна единице.

### Критерии интерпретации
Замена считается допустимой, если вносимая ею относительная погрешность целевой величины меньше погрешности измерения. Целевой величиной здесь является не сам импеданс, а его изменение за сердечный цикл.

**Входные данные.** Геометрия монтажа модели сердца: половина токовой базы $a=90$ мм, половина потенциальной базы $b=45$ мм ([01](01_Математическая_модель_импеданса.md), §1.3). Удельное сопротивление крови $\rho_2=1.35$ Ом·м — закреплено литературой и в число неизвестных не входит. Значения базового импеданса трансторакального монтажа из трёх независимых источников ([17](17_Дрейф_опорного_канала_и_порядок_съёмки.ipynb), §4.2).

**Допущения.** Прямая задача для сферы в однородном полупространстве берётся в виде (18.1) — ряда по полиномам Лежандра:

$$\delta Z=\frac{\rho_1}{\pi\,r_0}\sum_{n=0}^{N}\left(\frac{R}{r_{\text{in}}}\right)^{n+1}\left(\frac{R}{r_0}\right)^{n}\frac{n\,(\rho_2-\rho_1)}{(n+1)\rho_2+n\rho_1}\,P_n(\cos\theta) \tag{18.1}$$

где $\delta Z$ — вклад одной пары электродов в возмущение импеданса, Ом; $\rho_1$ — удельное сопротивление мягких тканей, Ом·м; $\rho_2$ — удельное сопротивление крови, Ом·м; $R$ — радиус сферы, м; $r_0$ и $r_{\text{in}}$ — расстояния от центра сферы до токового и потенциального электрода, м; $\theta$ — угол при центре сферы между направлениями на эти электроды; $P_n$ — полином Лежандра степени $n$; $N$ — число удерживаемых членов. Член $n=0$ обращается в нуль тождественно, что выражает сохранение тока ([01](01_Математическая_модель_импеданса.md), §3.2.1).

Удельное сопротивление мягких тканей вычисляется обращением однослойной формулы, (18.2):

$$\rho_1=\frac{Z_{\text{баз}}}{K},\qquad K=\frac{2b}{\pi\,(a^2-b^2)} \tag{18.2}$$

где $Z_{\text{баз}}$ — измеренный базовый импеданс, Ом; $K$ — геометрический множитель монтажа, 1/м.

In [1]:
# @title §1 Источник базового импеданса и его влияние на восстановленный радиус
import numpy as np
from scipy.special import eval_legendre
from scipy.optimize import brentq

A0, B0 = 0.090, 0.045          # полубазы монтажа модели сердца, м
RHO2   = 1.35                  # кровь, Ом·м — закреплена литературой
NTERMS = 20
K_GEOM = 2*B0/(np.pi*(A0**2 - B0**2))

def rho1_of(Z_base):
    """Обращение однослойной формулы (18.2)."""
    return Z_base/K_GEOM

def dZ_terms(rho1, R, h, rho2=RHO2, N=NTERMS):
    """Вклад каждого члена ряда (18.1) в полное возмущение, сфера по центру."""
    RA = RB = np.sqrt((R+h)**2 + A0**2)
    RM = RN = np.sqrt((R+h)**2 + B0**2)
    cs = lambda p, q, d: (p**2 + q**2 - d**2)/(2*p*q)
    n  = np.arange(N+1)
    coef = n*(rho2 - rho1)/((n+1)*rho2 + n*rho1)
    out = np.zeros(N+1)
    for r0, r_in, d, sgn in [(RA, RM, A0-B0, +1), (RA, RN, A0+B0, -1),
                             (RB, RN, A0-B0, +1), (RB, RM, A0+B0, -1)]:
        out += sgn*rho1/(np.pi*r0)*(R/r_in)**(n+1)*(R/r0)**n*coef*eval_legendre(n, cs(r0, r_in, d))
    return out

def dZ(rho1, R, h, rho2=RHO2): return float(dZ_terms(rho1, R, h, rho2).sum())
def Zbase(rho1, R, h, rho2=RHO2): return rho1*K_GEOM + dZ(rho1, R, h, rho2)

SRC = [("БГТУ, канал 1 (неисправен)", 12.703),
       ("РНЦХ, оба канала включены",  54.400),
       ("РНЦХ, канал 2 отключён",    101.000),
       ("Расчёт МКЭ (COMSOL)",       100.000)]
R0, H0 = 0.040, 0.020          # рабочая точка: радиус и глубина залегания, м

print("Геометрический множитель монтажа K = 2b/(π(a²−b²)) = %.3f 1/м" % K_GEOM)
print()
print("%-30s | %7s | %9s | %11s | %13s" % ("источник Z_баз", "Z, Ом", "ρ1, Ом·м", "∂Z/∂R, Ом/мм", "ΔR при 0.5 %"))
print("-"*88)
for name, z in SRC:
    r1 = rho1_of(z)
    Zb = Zbase(r1, R0, H0)
    d  = (Zbase(r1, R0+1e-5, H0) - Zbase(r1, R0-1e-5, H0))/2e-5/1000
    Rs = brentq(lambda R: Zbase(r1, R, H0) - (Zb - 0.005*Zb), 0.005, 0.075)
    print("%-30s | %7.2f | %9.3f | %+11.5f | %10.3f мм" % (name, z, r1, d, (Rs-R0)*1000))

Геометрический множитель монтажа K = 2b/(π(a²−b²)) = 4.716 1/м

источник Z_баз                 |   Z, Ом |  ρ1, Ом·м | ∂Z/∂R, Ом/мм |  ΔR при 0.5 %
----------------------------------------------------------------------------------------
БГТУ, канал 1 (неисправен)     |   12.70 |     2.694 |    -0.02317 |      2.609 мм
РНЦХ, оба канала включены      |   54.40 |    11.536 |    -0.27256 |      0.875 мм
РНЦХ, канал 2 отключён         |  101.00 |    21.418 |    -0.58248 |      0.744 мм
Расчёт МКЭ (COMSOL)            |  100.00 |    21.206 |    -0.57575 |      0.745 мм


**Анализ результатов.** Геометрический множитель монтажа равен 4.716 1/м. Значения удельного сопротивления мягких тканей, получаемые обращением (18.2), различаются между источниками в восемь раз: 2.694 Ом·м по каналу 1 прибора БГТУ против 21.418 Ом·м по измерению исправного прибора при отключённом соседнем канале.

Последствия для обратной задачи двоякие. Чувствительность импеданса к радиусу меняется в двадцать пять раз, от −0.023 до −0.582 Ом/мм. Изменение радиуса, восстановленное из одного и того же относительного пульсового изменения импеданса 0.5 %, составляет 2.609 мм при использовании неисправного канала против 0.744 мм при использовании исправного, то есть завышается в 3.5 раза.

Значения, полученные из измерения исправного прибора при отключённом соседнем канале (0.744 мм) и из расчёта МКЭ (0.745 мм), совпадают, что и ожидается: обе величины относятся к одиночной токовой паре. Значение при обоих включённых каналах (0.875 мм) отличается на 18 %, поскольку включает вклад межканального шунта ([17](17_Дрейф_опорного_канала_и_порядок_съёмки.ipynb), §4.1).

**Результаты и умозаключения.** Для модели сердца следует использовать базовый импеданс, измеренный **исправным прибором при отключённом соседнем канале**, поскольку аналитическая модель содержит одну токовую пару и второго генератора не описывает. Использование канала 1 прибора БГТУ приводит к завышению восстановленного изменения радиуса в 3.5 раза.

## §2. Отсутствие вырожденности сферической модели по общему множителю

В лёгочной двуслойной задаче ошибка усиления измерительного тракта тождественно неотличима от одновременного изменения обоих удельных сопротивлений ([09](09_Статическая_оценка_параметров.ipynb), §3.2, формула 9.12), поскольку оба параметра свободны. В сферической модели положение иное: $\rho_2$ закреплено литературным значением проводимости крови и в число неизвестных не входит. Проверяется, сохраняется ли вырожденность.

In [2]:
# @title §2 Проверка вырожденности при закреплённом ρ2
r1_ok = rho1_of(101.0)
print("%5s | %-28s | %-28s" % ("γ", "dZ(γρ1, ρ2=1.35) — как есть", "dZ(γρ1, γρ2) — оба масштаба"))
print("-"*66)
for g in (1.0, 1.5, 2.0):
    print("%5.1f | %+28.5f | %+28.5f" % (g, dZ(g*r1_ok, R0, H0, RHO2), dZ(g*r1_ok, R0, H0, g*RHO2)))
print()
print("Второй столбец воспроизводит γ·dZ(ρ1,ρ2) — однородность сохраняется, когда масштабируются оба.")
print("Первый столбец от него отличается, то есть при закреплённом ρ2 вырожденности НЕТ.")

    γ | dZ(γρ1, ρ2=1.35) — как есть  | dZ(γρ1, γρ2) — оба масштаба 
------------------------------------------------------------------
  1.0 |                    -14.27893 |                    -14.27893
  1.5 |                    -22.70049 |                    -21.41839
  2.0 |                    -31.17023 |                    -28.55786

Второй столбец воспроизводит γ·dZ(ρ1,ρ2) — однородность сохраняется, когда масштабируются оба.
Первый столбец от него отличается, то есть при закреплённом ρ2 вырожденности НЕТ.


**Анализ результатов.** При одновременном масштабировании обоих удельных сопротивлений возмущение умножается ровно на тот же множитель, что воспроизводит однородность прямой задачи первой степени. При закреплённом $\rho_2$ значения расходятся: множителю 2 отвечает −31.170 против −28.558 Ом.

**Результаты и умозаключения.** Ошибка калибровки прибора в сферической модели **не поглощается** переопределением параметров и потому прямо смещает восстановленный радиус. Это отличает её от лёгочной задачи и означает две вещи: модель уязвима к неверному базовому импедансу (§1), но она же в принципе способна обнаружить ошибку калибровки, чего двуслойная модель не может.

## §3. Мультипольный состав возмущения

Вопрос о допустимости замены реального сердца сферой сводится к тому, какими свойствами включения определяется наблюдаемая величина. Ряд (18.1) даёт прямой ответ: следует вычислить вклад каждого члена.

Дипольный член $n=1$ для сферы пропорционален $R^3$, то есть объёму. Если бы он исчерпывал сумму, равнообъёмная сфера была бы эквивалентна по импедансу автоматически. Члены более высокого порядка зависят от формы включения существеннее и такой пропорциональности не имеют.

In [3]:
# @title §3 Вклад мультипольных членов в полное возмущение
print("%6s %6s | %10s | %9s | %8s %8s %8s" %
      ("R, мм", "h, мм", "R/(R+h)", "dZ, Ом", "n=1", "n≤2", "n≤3"))
print("-"*70)
for R, h in [(0.040, 0.020), (0.040, 0.040), (0.030, 0.040), (0.030, 0.060)]:
    t = dZ_terms(rho1_of(101.0), R, h); tot = t.sum(); cum = np.cumsum(t)/tot*100
    print("%6.0f %6.0f | %10.2f | %+9.4f | %7.1f%% %7.1f%% %7.1f%%"
          % (R*1000, h*1000, R/(R+h), tot, cum[1], cum[2], cum[3]))
print()
print("Дипольный член даёт основную, но не подавляющую долю; квадрупольный вносит около 20 %.")

 R, мм  h, мм |    R/(R+h) |    dZ, Ом |      n=1      n≤2      n≤3
----------------------------------------------------------------------
    40     20 |       0.67 |  -14.2789 |    77.1%    98.0%    99.9%
    40     40 |       0.50 |   -5.6167 |    77.6%    97.6%    99.9%
    30     40 |       0.43 |   -3.3705 |    86.2%    99.2%   100.0%
    30     60 |       0.33 |   -1.3541 |    87.2%    99.2%   100.0%

Дипольный член даёт основную, но не подавляющую долю; квадрупольный вносит около 20 %.


**Анализ результатов.** В рабочей точке (радиус 40 мм, глубина залегания 20 мм) дипольный член даёт 77.1 % полного возмущения, сумма до квадрупольного включительно — 98.0 %, до октупольного — 99.9 %. При увеличении глубины доля диполя меняется слабо: 77.6 % при глубине 40 мм и 86.2 % при радиусе 30 мм и той же глубине.

Существенно отношение $R/(R+h)$, определяющее скорость убывания членов ряда. В рабочей точке оно равно 0.67: сфера крупнее собственной глубины залегания, и включение находится не в дальней зоне. Именно поэтому квадрупольный член не пренебрежим.

**Результаты и умозаключения.** Наблюдаемая величина на 77–86 % определяется дипольным откликом включения и примерно на 20 % — квадрупольным. Отсюда два независимых механизма расхождения при замене реального сердца сферой: несовпадение дипольного отклика, разбираемое в §4, и несовпадение квадрупольного, для которого равнообъёмная сфера не даёт вообще никакого приближения.

## §4. Почему геометрическая эквивалентность не влечёт импедансной

Дипольный отклик включения произвольной формы описывается тензором поляризуемости. Для эллипсоида с полуосями $a_1,a_2,a_3$ и проводимостями $\sigma_1$ снаружи и $\sigma_2$ внутри поляризуемость вдоль оси $i$ равна (18.3):

$$\alpha_i=V\,\frac{\sigma_2-\sigma_1}{\sigma_1+L_i\,(\sigma_2-\sigma_1)} \tag{18.3}$$

где $V$ — объём включения, м³; $L_i$ — коэффициент деполяризации вдоль оси $i$, безразмерный, $\sum_i L_i=1$; для шара $L_i=1/3$ при всех $i$.

Из (18.3) видно главное: поляризуемость пропорциональна объёму лишь при фиксированной форме. При $L_i\neq 1/3$ множитель при $V$ меняется, причём тем сильнее, чем выше контраст $\sigma_2/\sigma_1$. Сердце вытянуто, а контраст крови и мягких тканей велик, поэтому эффект нужно оценить численно.

In [4]:
# @title §4 Поляризуемость вытянутого эллипсоида против равнообъёмной сферы
def L_prolate(k):
    """Коэффициент деполяризации вдоль длинной оси вытянутого сфероида с отношением осей k."""
    e = np.sqrt(1.0 - 1.0/k**2)
    return (1 - e**2)/e**2*(1/(2*e)*np.log((1 + e)/(1 - e)) - 1)

rho1_ok = rho1_of(101.0)
s1, s2 = 1.0/rho1_ok, 1.0/RHO2          # проводимости, См/м
alpha = lambda L: (s2 - s1)/(s1 + L*(s2 - s1))    # на единицу объёма

print("Контраст проводимостей σ_кровь/σ_ткань = %.1f  (ρ1=%.2f, ρ2=%.2f Ом·м)"
      % (s2/s1, rho1_ok, RHO2))
print()
print("%-34s | %8s | %12s | %14s | %12s" %
      ("форма при том же объёме", "L", "α/V", "α/α_сферы", "R_экв/R_объём"))
print("-"*94)
a_sph = alpha(1/3.)
print("%-34s | %8.4f | %12.4f | %14s | %12s" % ("шар", 1/3., a_sph, "1.000", "1.000"))
for k in (1.5, 2.0, 2.5):
    Lz = L_prolate(k); Lx = (1 - Lz)/2
    for L, ax in [(Lz, "вдоль длинной оси"), (Lx, "поперёк длинной оси")]:
        r = alpha(L)/a_sph
        print("%-34s | %8.4f | %12.4f | %14.3f | %12.3f"
              % ("сфероид %.1f:1, %s" % (k, ax), L, alpha(L), r, r**(1/3.)))
print()
print("R_экв/R_объём — во сколько раз радиус импедансно-эквивалентной сферы")
print("отличается от радиуса равнообъёмной при том же дипольном отклике.")

Контраст проводимостей σ_кровь/σ_ткань = 15.9  (ρ1=21.42, ρ2=1.35 Ом·м)

форма при том же объёме            |        L |          α/V |      α/α_сферы | R_экв/R_объём
----------------------------------------------------------------------------------------------
шар                                |   0.3333 |       2.4962 |          1.000 |        1.000
сфероид 1.5:1, вдоль длинной оси   |   0.2330 |       3.3305 |          1.334 |        1.101
сфероид 1.5:1, поперёк длинной оси |   0.3835 |       2.2184 |          0.889 |        0.961
сфероид 2.0:1, вдоль длинной оси   |   0.1736 |       4.1522 |          1.663 |        1.185
сфероид 2.0:1, поперёк длинной оси |   0.4132 |       2.0812 |          0.834 |        0.941
сфероид 2.5:1, вдоль длинной оси   |   0.1351 |       4.9403 |          1.979 |        1.256
сфероид 2.5:1, поперёк длинной оси |   0.4324 |       2.0012 |          0.802 |        0.929

R_экв/R_объём — во сколько раз радиус импедансно-эквивалентной сферы
отличается от рад

**Анализ результатов.** При удельных сопротивлениях 21.42 Ом·м для мягких тканей и 1.35 Ом·м для крови контраст проводимостей составляет 15.9. Для вытянутого сфероида с отношением осей 2:1 коэффициент деполяризации вдоль длинной оси равен 0.1736, поперёк — 0.4132, тогда как у шара 1/3 при всех направлениях.

Дипольный отклик на единицу объёма при этом отличается от шарового в 1.664 раза вдоль длинной оси и в 0.834 раза поперёк. В пересчёте на радиус равнообъёмной сферы это даёт множители 1.185 и 0.941, то есть радиус импедансно-эквивалентной сферы отличается от геометрически эквивалентной на +18.5 % либо −5.9 % в зависимости от ориентации сердца относительно линии электродов. При отношении осей 2.5:1 разброс расширяется.

Для сопоставления: искомое систолическое изменение радиуса составляет около 0.74 мм при радиусе 40 мм, то есть 1.9 %. Погрешность замены на порядок превышает измеряемую величину.

**Результаты и умозаключения.** Равенство объёмов не обеспечивает равенства импеданса, и расхождение зависит от ориентации включения. Вместе с квадрупольным вкладом (§3) это означает, что сфера, построенная по геометрическому признаку, непригодна как замена реального сердца в задаче восстановления изменения объёма. Требуется иное определение эквивалентности.

## §5. Что уже реализовано в коде

В папке `Tikhomirov/cardio-model-py` есть модуль `sphere_fit.py` — порт `Kernel/part1/SphereMovingFunction.m`. Он предлагает три различных построения эквивалентной сферы:

| Функция | Критерий | Замечание |
|---|---|---|
| `eq_sph_nm_center_radius` | минимум $\sum_i(\lvert p_i-c\rvert-r)^2$ по точкам контура | подгонка окружности к границе, метод Нелдера — Мида |
| `circle_radius_by_contour_sqr` | $r=\sqrt{A/\pi}$ | равенство площади сечения |
| `req_for_heart` | из объёма по МРТ | см. ниже |

Три оговорки к повторному использованию.

Во-первых, весь модуль работает с **двумерным контуром**, а не с трёхмерной поверхностью. Для сегментации по томографии его придётся писать заново, а не переносить.

Во-вторых, три перечисленных критерия дают три разных радиуса, и обоснования выбора между ними в модуле нет.

В-третьих, `req_for_heart` реализует формулу $r=\sqrt[3]{2\cdot 3V/(0{,}6\cdot 4\pi)}\cdot 10$, что в $\sqrt[3]{2/0{,}6}=1{,}494$ раза больше радиуса равнообъёмной сферы. Множители 2 и 0,6 в порте не объяснены и в исходном тексте комментариев не имеют. До восстановления их происхождения функция непригодна.

**Незавершённая эллиптическая ветка.** В `WolframMath-master/Kernel/core/VolumeCalc.m` присутствует попытка перейти от круговых сечений к эллиптическим, оставшаяся незаконченной: в переключателе `SimpsonMethod` ветви `"Circle"` и `"Ellipse"` вызывают одну и ту же функцию `CutKonusCircleVolume`, а объявленная рядом `CutKonusEllipseVolume` имеет тождественную формулу и нигде не используется. Формула усечённого конуса с эллиптическим сечением требует двух диаметров на сечение, тогда как здесь передаётся один, поэтому ветвь не могла бы работать и в принципе. Это следует учитывать: эллиптическое приближение в проекте **не реализовано**, хотя его следы присутствуют.

Отдельного внимания требует `cardio-model-py/src/cardio_model/models.py`, где та же по виду формула $Z=\rho_1\,2b/(\pi(a^2-b^2))$ снабжена иной трактовкой: `a` названа большой полуосью эллипса торакального сегмента, а не половиной расстояния между токовыми электродами. Совпадает ли это с выводом, принятым в [01](01_Математическая_модель_импеданса.md), по имеющимся комментариям установить нельзя; перед использованием модуля трактовку следует сверить с исходным выводом.

## §6. Операциональное определение и порядок проверки

Из §3 и §4 следует, что эквивалентность должна определяться **через прямую задачу**, а не через геометрию. Предлагаемое определение: эквивалентной называется сфера, воспроизводящая расчёт по реальной геометрии в том смысле, который важен для цели работы.

Существенно, что целью является восстановление **изменения** объёма, а не абсолютного импеданса. Одна сфера не может быть эквивалентна одновременно по $Z$ и по $\partial Z/\partial V$, поэтому критерий выбирается осознанно, и для настоящей задачи это производная (18.4):

$$\min_{R,\,h,\,x,\,y}\ \sum_{m}w_m\left[\left(\frac{\partial Z}{\partial V}\right)^{\text{сфера}}_{m}-\left(\frac{\partial Z}{\partial V}\right)^{\text{МКЭ}}_{m}\right]^2 \tag{18.4}$$

где $m$ — номер монтажа из набора, по которому ведётся согласование; $w_m$ — вес монтажа, безразмерный; производные вычисляются при одинаковом относительном изменении объёма камер.

Порядок работ.

1. Взять сегментированную геометрию сердца из пайплайна `3D_Slicer`.
2. Рассчитать МКЭ (`MATLAB_TRKG4_real_subjects`) для реального монтажа и получить $Z$; повторить с объёмом камер, изменённым на несколько процентов, и получить $\partial Z/\partial V$.
3. Подобрать параметры сферы по критерию (18.4).
4. Отдельно построить геометрически эквивалентную сферу и сопоставить её с импедансно эквивалентной. Расхождение между ними — количественная мера адекватности сферического приближения и самостоятельный результат.

Ограничение, которое следует учитывать при планировании. У сферы четыре параметра, а один монтаж даёт одно число, поэтому согласование требует набора монтажей. Из расчёта МКЭ виртуальные монтажи можно получать в любом количестве, но информативность их различна, и план следует строить, проверяя обусловленность матрицы чувствительности к четырём параметрам, а не назначать монтажи произвольно.

## §7. Выводы

1. **Источник базового импеданса определяет масштаб ответа.** Удельное сопротивление мягких тканей, получаемое обращением однослойной формулы, различается в восемь раз между каналом 1 прибора БГТУ и измерением исправного прибора. Восстановленное изменение радиуса при этом завышается в 3.5 раза, а чувствительность импеданса к радиусу занижается в двадцать пять раз. Следует использовать измерение **исправным прибором при отключённом соседнем канале**, поскольку аналитическая модель содержит одну токовую пару.

2. **Вырожденности по общему множителю в сферической модели нет**, так как $\rho_2$ закреплено литературой. Ошибка калибровки прибора не поглощается переопределением параметров и прямо смещает результат; тем самым модель уязвима к неверному базовому импедансу, но способна обнаружить ошибку калибровки.

3. **Дипольный член даёт 77–86 % возмущения, квадрупольный около 20 %.** Отношение радиуса к глубине залегания в рабочей точке равно 0.67, то есть включение не находится в дальней зоне, и обрывать ряд на диполе нельзя.

4. **Равенство объёмов не обеспечивает равенства импеданса.** Дипольный отклик определяется поляризуемостью, зависящей от формы через коэффициенты деполяризации. Для вытянутого сфероида 2:1 при контрасте проводимостей 15.9 радиус импедансно эквивалентной сферы отличается от равнообъёмной на +18.5 % вдоль длинной оси и −5.9 % поперёк, тогда как искомое систолическое изменение радиуса составляет 1.9 %. Погрешность замены на порядок превышает измеряемую величину.

5. **Эквивалентность следует определять через прямую задачу** и согласовывать производную $\partial Z/\partial V$, а не абсолютный импеданс (18.4). Расхождение геометрически и импедансно эквивалентных сфер — самостоятельный результат, характеризующий адекватность сферического приближения.

6. **Имеющийся код решает другую задачу.** `sphere_fit.py` работает с двумерным контуром, предлагает три несогласованных критерия и содержит функцию `req_for_heart` с необъяснёнными множителями, дающую радиус в 1.494 раза больше равнообъёмного. Эллиптическое приближение в проекте не реализовано: в `VolumeCalc.m` соответствующая ветвь вызывает круговую функцию, а объявленная эллиптическая формула тождественна круговой и не используется.